# Proximal Policy Optimization (PPO)

https://spinningup.openai.com/en/latest/algorithms/ppo.html
https://docs.pytorch.org/rl/stable/tutorials/coding_ppo.html

In [1]:
import torch
from tensordict.nn import NormalParamExtractor, TensorDictModule
from torch import nn
from torchrl import logger
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import Compose, DoubleToFloat, GymEnv, StepCounter, TransformedEnv
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:574: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  PUCT = functools.partial(PUCTScore, c=5)  # AlphaGo default value
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:575: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB = functools.partial(UCBScore, c=math.sqrt(2))  # default from Auer et al. 2002
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:576: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB1_TUNED = functools.partial(
/usr/local/Caskroom/mini

In [2]:
env = TransformedEnv(
    GymEnv("InvertedPendulum-v5"),
    Compose([DoubleToFloat(), StepCounter()]),
)

In [3]:
_ = env.set_seed(0)
_ = torch.manual_seed(0)

PPO uses a stochastic policy which samples from a Normal distribution.
The distribution parameters (`loc` and `scale`) are the outputs of the policy network.
The `TanhNormal` distribution uses the tanh function to ensure that the sample range matches
the action spec determined by the environment.

In [4]:
NUM_CELLS = 128

# maps observations to parameters (loc, scale) of a Normal distribution
policy_net = nn.Sequential(
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(2),
    NormalParamExtractor(),
)

# stochastic policy which samples actions from the TanhNormal distribution
policy_module = ProbabilisticActor(
    module=TensorDictModule(
        policy_net, in_keys=["observation"], out_keys=["loc", "scale"]
    ),
    spec=env.action_spec,
    in_keys=["loc", "scale"],
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": env.action_spec.space.low.item(),  # type: ignore
        "high": env.action_spec.space.high.item(),  # type: ignore
    },
    return_log_prob=True,
)

The value network maps observations `s` to the (current estimate of) the value `V(s)`.

In [5]:
value_net = nn.Sequential(
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(NUM_CELLS),
    nn.Tanh(),
    nn.LazyLinear(1),
)

# TensorDictModule with in_keys=["observation"] and out_keys=["state_value"]
value_module = ValueOperator(value_net)

In [6]:
_ = value_module(env.rollout(5, policy_module))  # initialize lazy layers

In [7]:
LEARNING_RATE = 1e-4

advantage_module = GAE(
    gamma=0.99,
    lmbda=0.95,
    value_network=value_module,
    average_gae=True,
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=0.2,
    entropy_bonus=False,
)

optimizer = torch.optim.Adam(loss_module.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 10_000)

PPO is an on-policy algorithm, meaning that we only use data obtained from the latest (current) policy when training.
It does not use a replay buffer in the proper sense, but rather only for convenience as we want to split data batches
into mini-batches to perform multiple optimization steps; the replay buffer is therefore used only as a shuffler/sampler
for the latest batch from the collector (hence why its size is the size of a single batch).

In [8]:
FRAMES_PER_BATCH = 100

collector = Collector(
    env,
    policy=policy_module,
    frames_per_batch=FRAMES_PER_BATCH,
    total_frames=-1,
)

buffer = ReplayBuffer(storage=LazyTensorStorage(max_size=FRAMES_PER_BATCH))

We use an auxiliary function to evaluate the policy during training.
Since the policy is stochastic by definition, we evaluate it in deterministic mode 
(i.e. choosing the action with highest probability instead of sampling from the distribution),
as this gives a non-random evaluation.

We evaluate on a single episode and measure the total number of steps, since the goal of the environment
is to keep the pole upright for as long as possible.

In [9]:
@torch.no_grad()
def evaluate_policy(env: TransformedEnv, policy_module: ProbabilisticActor):
    env.reset()

    with set_exploration_type(ExplorationType.DETERMINISTIC):
        rollout = env.rollout(1000, policy_module)

    env.reset()

    return rollout["next", "step_count"].max().item()

We train for multiple epochs on each batch of data for efficiency.
At each epoch, the data is split into mini-batches and an optimization step is taken for each mini-batch.
Success is defined as having attained an episode with at least 200 steps.

In [10]:
NUM_EPOCHS = 10
MINI_BATCH_SIZE = 64


step_count = 0
episode_count = 0

for idx, data in enumerate(collector, start=1):
    step_count += data.numel()
    episode_count += data["next", "done"].sum()

    max_steps = evaluate_policy(env, policy_module)

    if max_steps > 200:
        break

    if idx % 10 == 0:
        logger.info(f"[{idx:>3}] max: {max_steps:>3}")

    # train on the entire batch multiple times
    for _ in range(NUM_EPOCHS):
        advantage_module(data)  # recompute GAE as it depends on the value module
        buffer.extend(data)  # replace previous data

        # partition batch into minibatches and take one optimizer step per minibatch
        for _ in range(FRAMES_PER_BATCH // MINI_BATCH_SIZE):
            mini_batch = buffer.sample(MINI_BATCH_SIZE)
            loss_values = loss_module(mini_batch)

            loss = loss_values["loss_objective"] + loss_values["loss_critic"]
            loss.backward()

            nn.utils.clip_grad_norm_(loss_module.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

    scheduler.step()


logger.info(f"solved after {step_count} steps, {episode_count} episodes")

2026-02-16 11:28:14,684 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100]) shape [END]
2026-02-16 11:28:17,724 [torchrl][INFO]    [ 10] max:  20 [END]
2026-02-16 11:28:21,053 [torchrl][INFO]    [ 20] max:  76 [END]
2026-02-16 11:28:25,077 [torchrl][INFO]    [ 30] max: 127 [END]
2026-02-16 11:28:27,122 [torchrl][INFO]    solved after 3400 steps, 182 episodes [END]
